# 🛩️ AeroLLM — Fine-tuning Phi-3.5-mini sur données ASRS

**QLoRA fine-tuning sur Google Colab T4 GPU**

- Modèle : microsoft/Phi-3.5-mini-instruct (3.8B)
- Dataset : 2000 rapports ASRS (NASA Aviation Safety Reporting System)
- Technique : QLoRA 4-bit + LoRA adapters
- Sauvegarde automatique sur HuggingFace Hub

In [ ]:
# Cellule 1 — Verification GPU
!nvidia-smi
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cellule 2 — Installation des dependances
!pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub rich

In [ ]:
# Cellule 3 — Clone du repo et preparation dataset
!git clone https://github.com/NaimMG/AeroLLM.git
%cd AeroLLM
!python src/prepare_data.py

In [ ]:
# Cellule 4 — Connexion HuggingFace (token Write requis)
from huggingface_hub import login
login()

In [ ]:
# Cellule 5 — Configuration
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'
OUTPUT_DIR = './outputs/aerollm-qlora'
HF_REPO = 'NaimMG/aerollm-phi3.5-mini-asrs'

print('Model:', MODEL_ID)
print('HF Repo:', HF_REPO)
print('Config OK')

In [ ]:
# Cellule 6 — Chargement et formatage du dataset
with open('data/processed/train.json', 'r') as f:
    data = json.load(f)

def format_prompt(item):
    return {
        'text': (
            '<|system|>\n'
            + item['instruction']
            + '<|end|>\n'
            + '<|user|>\n'
            + item['input']
            + '<|end|>\n'
            + '<|assistant|>\n'
            + item['output']
            + '<|end|>'
        )
    }

split = int(len(data) * 0.9)
train_data = Dataset.from_list([format_prompt(d) for d in data[:split]])
eval_data = Dataset.from_list([format_prompt(d) for d in data[split:]])

print('Train:', len(train_data), 'exemples')
print('Eval:', len(eval_data), 'exemples')
print('Apercu prompt:')
print(train_data[0]['text'][:300])

In [ ]:
# Cellule 7 — Chargement modele en 4-bit QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
model.config.use_cache = False

print('Modele charge en 4-bit QLoRA')

In [ ]:
# Cellule 8 — Configuration LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cellule 9 — Fine-tuning
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    evaluation_strategy='steps',
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=HF_REPO,
    hub_strategy='checkpoint',
    report_to='none',
    warmup_ratio=0.03,
    lr_scheduler_type='cosine'
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer
)

print('Debut du fine-tuning...')
print('Duree estimee sur T4 : 45-60 minutes')
trainer.train()
print('Fine-tuning termine!')

In [ ]:
# Cellule 10 — Sauvegarde sur HuggingFace Hub
trainer.push_to_hub()
tokenizer.push_to_hub(HF_REPO)
print('Modele publie : https://huggingface.co/' + HF_REPO)

In [ ]:
# Cellule 11 — Test du modele fine-tune
test_input = (
    'Aeronef : B737-800\n'
    'Phase de vol : Final Approach\n'
    'Rapport : During final approach runway 28L, GPWS terrain warning activated '
    'at 500ft AGL. Crew executed go-around immediately. '
    'Investigation revealed incorrect QNH setting by crew during pre-flight.'
)

prompt = (
    '<|system|>\n'
    'Tu es un expert en securite aeronautique certifie FAA/EASA. '
    'Analyse ce rapport incident ASRS.<|end|>\n'
    '<|user|>\n'
    + test_input
    + '<|end|>\n'
    '<|assistant|>\n'
)

inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)
print('=== REPONSE DU MODELE FINE-TUNE ===')
print(response)